# Test All Models on New Data

This notebook loads the **test images** from `Test_Data/` and runs all three trained models on them:

| Test Image | Resolution | Model | Architecture |
|-----------|-----------|-------|-------------|
| `test_1_6mc1-c4` (RGB+NIR) | ~1.6 m | U-Net GBNM | Aerial RGB 6-class |
| `test_1m` (Panchromatic) | 1 m | ResNet-18 Cartosat | Patch classifier 6-class |
| `test_2_5m` (Panchromatic) | 2.5 m | U-Net Vashi | Sentinel-2 7-class (band-replicated) |

For each model we:
1. Load and preprocess the test image
2. Run inference
3. Visualise: RGB / Segmentation / Overlay
4. Print class area statistics
5. Export georeferenced vector shapefiles

In [80]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
import rasterio
from rasterio.transform import Affine
import torch
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = os.getcwd()
if os.path.exists(os.path.join(NOTEBOOK_DIR, 'WebPlatform')):
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

WEB_DIR = os.path.join(PROJECT_ROOT, 'WebPlatform')
if WEB_DIR not in sys.path:
    sys.path.insert(0, WEB_DIR)

print(f'Project root: {PROJECT_ROOT}')
print(f'WebPlatform:  {WEB_DIR}')

from model_utils import load_unet_vashi, load_unet_gbnm, load_cartosat, MODEL_PATHS
from inference_utils import run_unet, run_cartosat
from export_utils import raster_to_vectors, vectors_to_shapefile_zip, pred_map_to_geotiff_bytes
from viz_utils import UNET_CLASS_NAMES, UNET_CLASS_COLORS, CARTOSAT_CLASS_NAMES, CARTOSAT_CLASS_COLORS
from metrics_utils import print_confidence_metrics, plot_entropy_and_confidence, plot_class_distribution_pie
from labeling_utils import (generate_kmeans_gt, generate_spectral_kmeans_gt,
    generate_texture_kmeans_gt, smooth_label_map,
    majority_vote_match, compute_full_metrics,
    print_accuracy_report, plot_confusion_matrix, plot_gt_vs_pred,
    plot_model_comparison, print_comparison_table, downsample_image)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

for name, path in MODEL_PATHS.items():
    status = 'FOUND' if os.path.isfile(path) else 'MISSING'
    print(f'  {name:15s}: {status}')

TEST_DIR = os.path.join(PROJECT_ROOT, 'Test_Model', 'Test_Data')
OUT_DIR  = os.path.join(PROJECT_ROOT, 'Test_Model', 'results')
TIFF_DIR = os.path.join(PROJECT_ROOT, 'Test_Model', 'geotiff')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TIFF_DIR, exist_ok=True)
print(f'\nTest data: {TEST_DIR}')
print(f'  Exists: {os.path.isdir(TEST_DIR)}')
print(f'Output:    {OUT_DIR}')
print(f'GeoTIFFs:  {TIFF_DIR}')

Project root: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)
WebPlatform:  c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\WebPlatform
Device: cuda
  unet_vashi     : FOUND
  unet_gbnm      : FOUND
  cartosat       : FOUND
  maskrcnn       : FOUND

Test data: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\Test_Data
  Exists: True
Output:    c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results
GeoTIFFs:  c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\geotiff


In [81]:
def pct_stretch(arr, lo=2, hi=98):
    v = arr[arr != 0] if (arr != 0).any() else arr.ravel()
    a, b = np.percentile(v, lo), np.percentile(v, hi)
    return np.clip((arr - a) / max(b - a, 1e-6), 0, 1)

def visualise_result(rgb, pred_map, class_names, class_colors, title, save_path=None):
    from PIL import Image as PILImage
    if rgb.shape[:2] != pred_map.shape:
        pil = PILImage.fromarray((rgb * 255).astype(np.uint8) if rgb.max() <= 1 else rgb.astype(np.uint8))
        pil = pil.resize((pred_map.shape[1], pred_map.shape[0]), PILImage.BILINEAR)
        rgb = np.array(pil).astype(np.float32) / 255.0
    n_cls = len(class_colors)
    lut = np.zeros((256, 3), dtype=np.float32)
    for i, c in enumerate(class_colors[:n_cls]):
        lut[i] = to_rgb(c)
    seg_rgb = lut[pred_map]
    overlay = 0.5 * rgb + 0.5 * seg_rgb
    fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor='#0a0e17')
    for ax, img, t in zip(axes, [rgb, seg_rgb, overlay],
            ['Original', 'Segmentation', 'Overlay (50% blend)']):
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(t, color='white', fontsize=12, fontweight='bold')
        ax.axis('off')
    patches = [mpatches.Patch(color=c, label=n) for c, n in zip(class_colors, class_names)]
    fig.legend(handles=patches, loc='lower center', ncol=min(7, len(class_names)),
               fontsize=9, framealpha=0.2, facecolor='#0a0e17',
               edgecolor='#1e4976', labelcolor='white', bbox_to_anchor=(0.5, -0.02))
    plt.suptitle(title, color='#38bdf8', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0a0e17')
        print(f'Saved: {save_path}')
    plt.show()

def print_class_stats(pred_map, class_names, res_m):
    total = pred_map.size
    hdr = 'Class                  Pixels Coverage    Area (m2) Area (km2)'
    print(hdr)
    print('-' * 65)
    for i, name in enumerate(class_names):
        cnt = int((pred_map == i).sum())
        pct = 100 * cnt / total
        area_m2 = cnt * (res_m ** 2)
        print(f'{name:<20s} {cnt:>10,} {pct:>7.1f}% {area_m2:>12,.0f} {area_m2/1e6:>10.4f}')

print('Helpers ready.')

Helpers ready.


---
## 1. Inspect Test Data

In [82]:
print('=' * 70)
print('  TEST DATA INSPECTION')
print('=' * 70)

test_files = {
    'test_1m':     'Panchromatic 1m (Cartosat)',
    'test_1_6mc1': 'Band 1 of 1.6m multispectral',
    'test_1_6mc2': 'Band 2 of 1.6m multispectral',
    'test_1_6mc3': 'Band 3 of 1.6m multispectral',
    'test_1_6mc4': 'Band 4 of 1.6m multispectral',
    'test_2_5m':   'Panchromatic 2.5m',
}

for name, desc in test_files.items():
    path = os.path.join(TEST_DIR, name)
    try:
        with rasterio.open(path) as src:
            d = src.read(1).astype('float32')
            print(f'\n{name}  ({desc})')
            print(f'  Size    : {src.width} x {src.height} px')
            print(f'  Bands   : {src.count}')
            print(f'  Dtype   : {src.dtypes[0]}')
            print(f'  Res     : {src.res[0]:.2f} m x {src.res[1]:.2f} m')
            print(f'  CRS     : {src.crs}')
            print(f'  Range   : {d.min():.0f} - {d.max():.0f}  mean={d.mean():.1f}')
    except Exception as e:
        print(f'\n{name}: CANNOT READ — {e}')

  TEST DATA INSPECTION



test_1m  (Panchromatic 1m (Cartosat))
  Size    : 4931 x 4020 px
  Bands   : 1
  Dtype   : int16
  Res     : 1.00 m x 1.00 m
  CRS     : EPSG:32643
  Range   : -32768 - 398  mean=-12074.8

test_1_6mc1  (Band 1 of 1.6m multispectral)
  Size    : 2220 x 1583 px
  Bands   : 1
  Dtype   : int16
  Res     : 0.00 m x 0.00 m
  CRS     : EPSG:4326
  Range   : -32768 - 2007  mean=-954.1

test_1_6mc2  (Band 2 of 1.6m multispectral)
  Size    : 2220 x 1583 px
  Bands   : 1
  Dtype   : int16
  Res     : 0.00 m x 0.00 m
  CRS     : EPSG:4326
  Range   : -32768 - 2535  mean=-875.9

test_1_6mc3  (Band 3 of 1.6m multispectral)
  Size    : 2220 x 1583 px
  Bands   : 1
  Dtype   : int16
  Res     : 0.00 m x 0.00 m
  CRS     : EPSG:4326
  Range   : -32768 - 2376  mean=-860.9

test_1_6mc4  (Band 4 of 1.6m multispectral)
  Size    : 2220 x 1583 px
  Bands   : 1
  Dtype   : int16
  Res     : 0.00 m x 0.00 m
  CRS     : EPSG:4326
  Range   : -32768 - 3581  mean=-722.4

test_2_5m  (Panchromatic 2.5m)
  Size 

---
## 1b. Convert ESRI Grid (ADF) → GeoTIFF
The web platform only accepts `.tif` files. This converts all ADF test data to GeoTIFF so you can upload them.

In [83]:
adf_datasets = {
    'test_1m':     'Cartosat panchromatic 1m',
    'test_1_6mc1': 'Band 1 of 1.6m multispectral',
    'test_1_6mc2': 'Band 2 of 1.6m multispectral',
    'test_1_6mc3': 'Band 3 of 1.6m multispectral',
    'test_1_6mc4': 'Band 4 (NIR) of 1.6m multispectral',
    'test_2_5m':   'Panchromatic 2.5m',
}

print('Converting ESRI Grid (ADF) -> GeoTIFF...\n')
converted = {}
for name, desc in adf_datasets.items():
    adf_path = os.path.join(TEST_DIR, name)
    tif_path = os.path.join(TIFF_DIR, f'{name}.tif')
    try:
        with rasterio.open(adf_path) as src:
            profile = src.profile.copy()
            profile.update(driver='GTiff', compress='lzw')
            data = src.read()
            with rasterio.open(tif_path, 'w', **profile) as dst:
                dst.write(data)
        size_kb = os.path.getsize(tif_path) / 1024
        converted[name] = tif_path
        print(f'  {name:15s} -> {os.path.basename(tif_path):25s} ({size_kb:.0f} KB)')
    except Exception as e:
        print(f'  {name:15s} FAILED: {e}')

# Create a 3-band RGB stack for the web platform
rgb_bands = ['test_1_6mc1', 'test_1_6mc2', 'test_1_6mc3']
if all(b in converted for b in rgb_bands):
    with rasterio.open(converted[rgb_bands[0]]) as ref:
        profile = ref.profile.copy()
        profile.update(count=3, driver='GTiff', compress='lzw')
    rgb_tif = os.path.join(TIFF_DIR, 'test_1_6m_rgb.tif')
    with rasterio.open(rgb_tif, 'w', **profile) as dst:
        for i, bname in enumerate(rgb_bands, 1):
            with rasterio.open(converted[bname]) as src:
                dst.write(src.read(1), i)
    print(f'\n  RGB stack     -> test_1_6m_rgb.tif             ({os.path.getsize(rgb_tif)/1024:.0f} KB)')

# 4-band stack (RGB+NIR) for Vashi U-Net
all_bands = ['test_1_6mc1', 'test_1_6mc2', 'test_1_6mc3', 'test_1_6mc4']
if all(b in converted for b in all_bands):
    with rasterio.open(converted[all_bands[0]]) as ref:
        profile = ref.profile.copy()
        profile.update(count=4, driver='GTiff', compress='lzw')
    rgba_tif = os.path.join(TIFF_DIR, 'test_1_6m_4band.tif')
    with rasterio.open(rgba_tif, 'w', **profile) as dst:
        for i, bname in enumerate(all_bands, 1):
            with rasterio.open(converted[bname]) as src:
                dst.write(src.read(1), i)
    print(f'  4-band stack  -> test_1_6m_4band.tif           ({os.path.getsize(rgba_tif)/1024:.0f} KB)')

print(f'\nAll GeoTIFFs saved to: {TIFF_DIR}')
print('You can now upload these .tif files in the Streamlit web platform.')

Converting ESRI Grid (ADF) -> GeoTIFF...

  test_1m         -> test_1m.tif               (10407 KB)
  test_1_6mc1     -> test_1_6mc1.tif           (4941 KB)
  test_1_6mc2     -> test_1_6mc2.tif           (4982 KB)
  test_1_6mc3     -> test_1_6mc3.tif           (4932 KB)
  test_1_6mc4     -> test_1_6mc4.tif           (5524 KB)
  test_2_5m       -> test_2_5m.tif             (2239 KB)

  RGB stack     -> test_1_6m_rgb.tif             (14855 KB)
  4-band stack  -> test_1_6m_4band.tif           (20378 KB)

All GeoTIFFs saved to: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\geotiff
You can now upload these .tif files in the Streamlit web platform.


---
## 2. Test: U-Net GBNM (Aerial RGB, 1.6m)

Stack `test_1_6mc1` + `test_1_6mc2` + `test_1_6mc3` into a 3-band RGB image and run the Aerial RGB U-Net.

In [84]:
print('Loading test_1_6m bands...')

bands_1_6m = []
geo_meta_1_6m = None

for band_name in ['test_1_6mc1', 'test_1_6mc2', 'test_1_6mc3']:
    path = os.path.join(TEST_DIR, band_name)
    with rasterio.open(path) as src:
        bands_1_6m.append(src.read(1).astype(np.float32))
        if geo_meta_1_6m is None:
            geo_meta_1_6m = {
                'crs_wkt':   src.crs.to_wkt() if src.crs else None,
                'transform': tuple(src.transform),
                'width':     src.width,
                'height':    src.height,
            }

# Stack as (H, W, 3) — format expected by inference pipeline
img_1_6m = np.stack(bands_1_6m, axis=-1)
print(f'Stacked image: {img_1_6m.shape}  dtype={img_1_6m.dtype}')
print(f'Range: {img_1_6m.min():.0f} - {img_1_6m.max():.0f}')
print(f'CRS: {geo_meta_1_6m["crs_wkt"][:60] if geo_meta_1_6m["crs_wkt"] else "None"}...')

# Display RGB preview
rgb_1_6m = np.stack([pct_stretch(img_1_6m[:,:,i]) for i in range(3)], axis=-1)
plt.figure(figsize=(10, 8))
plt.imshow(rgb_1_6m)
plt.title('Test Image — 1.6m RGB (3-band stack)', fontsize=13)
plt.axis('off')
plt.show()

Loading test_1_6m bands...
Stacked image: (1583, 2220, 3)  dtype=float32
Range: -32768 - 2535
CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...


In [85]:
# Load model
print('Loading U-Net GBNM model...')
model_gbnm, ckpt_gbnm = load_unet_gbnm()
print(f'  Classes: {ckpt_gbnm.get("num_classes")}  Input channels: {ckpt_gbnm.get("in_channels")}')

# Run inference
print('Running inference...')
result_gbnm = run_unet(model_gbnm, ckpt_gbnm, img_1_6m.copy(), 'unet_gbnm')

pred_gbnm = result_gbnm['pred_map']
conf_gbnm = result_gbnm['conf_map']
ds_step   = result_gbnm.get('downsample_step', 1)
print(f'Prediction shape: {pred_gbnm.shape}  (downsample_step={ds_step})')
print(f'Unique classes predicted: {np.unique(pred_gbnm)}')
print(f'Mean confidence: {conf_gbnm.mean()*100:.1f}%')

Loading U-Net GBNM model...
  Classes: 6  Input channels: 3
Running inference...
Prediction shape: (792, 1110)  (downsample_step=2)
Unique classes predicted: [0 1 2 3 4 5]
Mean confidence: 70.9%


In [86]:
# Visualise
names_gbnm  = result_gbnm['class_names']
colors_gbnm = result_gbnm['class_colors']

visualise_result(
    rgb_1_6m, pred_gbnm, names_gbnm, colors_gbnm,
    'U-Net GBNM — Aerial RGB Test (1.6m)',
    save_path=os.path.join(OUT_DIR, 'test_1_6m_unet_gbnm.png')
)

print('\nClass Area Statistics (res=1.6m):')
print_class_stats(pred_gbnm, names_gbnm, res_m=1.6)

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1_6m_unet_gbnm.png

Class Area Statistics (res=1.6m):
Class                  Pixels Coverage    Area (m2) Area (km2)
-----------------------------------------------------------------
Background                3,800     0.4%        9,728     0.0097
Water                   143,917    16.4%      368,428     0.3684
Vegetation              592,604    67.4%    1,517,066     1.5171
Roads                    45,575     5.2%      116,672     0.1167
Building                 19,636     2.2%       50,268     0.0503
Open Ground              73,588     8.4%      188,385     0.1884


In [87]:
# Confidence metrics for U-Net GBNM
print_confidence_metrics(pred_gbnm, conf_gbnm, names_gbnm, colors_gbnm)
plot_entropy_and_confidence(pred_gbnm, conf_gbnm, names_gbnm, colors_gbnm,
    'U-Net GBNM Confidence Analysis (1.6m RGB)',
    save_path=os.path.join(OUT_DIR, 'test_1_6m_gbnm_confidence.png'))
plot_class_distribution_pie(pred_gbnm, names_gbnm, colors_gbnm,
    'U-Net GBNM Class Distribution')

  CONFIDENCE METRICS (based on softmax probabilities)

Class                  Pixels  Mean Conf     Min     Max     Std    <50%    <70%
--------------------------------------------------------------------------------
Background              3,800      16.7%   16.7%   16.7%    0.0%  100.0%  100.0%
Water                 143,917      70.2%   19.5%  100.0%   22.8%   25.2%   49.0%
Vegetation            592,604      76.5%   21.2%  100.0%   17.6%   10.4%   32.4%
Roads                  45,575      48.1%   20.9%  100.0%   16.5%   66.4%   89.1%
Building               19,636      46.5%   20.7%  100.0%   16.9%   69.6%   90.3%
Open Ground            73,588      50.6%   21.2%  100.0%   15.9%   56.6%   87.0%

Overall mean confidence: 70.9%
Pixels below 50% confidence: 21.3%
Pixels below 70% confidence: 44.2%
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1_6m_gbnm_confidence.png


In [88]:
# Export vectors
print('Generating vector shapefile...')
gdf_gbnm = raster_to_vectors(
    pred_gbnm, names_gbnm, colors_gbnm,
    geo_meta=geo_meta_1_6m, downsample_step=ds_step
)
print(f'Total polygons: {len(gdf_gbnm)}')
print(gdf_gbnm.groupby('class_name').size().to_string())

shp_path = os.path.join(OUT_DIR, 'test_1_6m_vectors.zip')
shp_bytes = vectors_to_shapefile_zip(gdf_gbnm)
with open(shp_path, 'wb') as f:
    f.write(shp_bytes)
print(f'Saved: {shp_path} ({len(shp_bytes)/1024:.0f} KB)')

# Save GeoTIFF
tiff_bytes = pred_map_to_geotiff_bytes(pred_gbnm, geo_meta=geo_meta_1_6m, downsample_step=ds_step)
tiff_path = os.path.join(OUT_DIR, 'test_1_6m_prediction.tif')
with open(tiff_path, 'wb') as f:
    f.write(tiff_bytes)
print(f'Saved: {tiff_path}')

Generating vector shapefile...
Total polygons: 57654
class_name
Building        6909
Open Ground    18372
Roads          20935
Vegetation      4196
Water           7242
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1_6m_vectors.zip (2511 KB)
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1_6m_prediction.tif


---
## 3. Test: Cartosat ResNet-18 (Panchromatic, 1m)

Load `test_1m` and run the Cartosat patch classifier.

In [89]:
print('Loading test_1m...')
with rasterio.open(os.path.join(TEST_DIR, 'test_1m')) as src:
    img_1m = src.read(1).astype(np.float32)
    geo_meta_1m = {
        'crs_wkt':   src.crs.to_wkt() if src.crs else None,
        'transform': tuple(src.transform),
        'width':     src.width,
        'height':    src.height,
    }

print(f'Shape: {img_1m.shape}  Range: {img_1m.min():.0f} - {img_1m.max():.0f}')
print(f'CRS: {src.crs}')

# Convert to (H, W, 1) for inference pipeline
img_1m_hwc = img_1m[:, :, np.newaxis]

# Display
plt.figure(figsize=(10, 8))
plt.imshow(pct_stretch(img_1m), cmap='gray')
plt.title('Test Image — 1m Panchromatic (Cartosat)', fontsize=13)
plt.axis('off')
plt.show()

Loading test_1m...
Shape: (4020, 4931)  Range: -32768 - 398
CRS: EPSG:32643


In [90]:
# Load model
print('Loading Cartosat ResNet-18 model...')
model_cart, ckpt_cart = load_cartosat()
print(f'  Classes: {len(ckpt_cart.get("class_names", []))}')
print(f'  Class names: {ckpt_cart.get("class_names")}')

# Run inference
print('Running inference...')
result_cart = run_cartosat(model_cart, ckpt_cart, img_1m_hwc.copy())

pred_cart = result_cart['pred_map']
conf_cart = result_cart['conf_map']
print(f'Prediction shape: {pred_cart.shape}')
print(f'Unique classes predicted: {np.unique(pred_cart)}')
print(f'Mean confidence: {conf_cart.mean()*100:.1f}%')

Loading Cartosat ResNet-18 model...
  Classes: 6
  Class names: ['Low_Vegetation_Fields', 'Beaches_Sea_Shores', 'Roads_and_Pavement', 'Urban_Built_Up', 'Deep_Water_or_Shadows', 'Bare_Earth_Exposed_Soil']
Running inference...
Prediction shape: (4020, 4931)
Unique classes predicted: [0 1 3]
Mean confidence: 83.1%


In [91]:
# Visualise
names_cart  = result_cart['class_names']
colors_cart = result_cart['class_colors']

# Make grayscale RGB for display
gray_disp = pct_stretch(img_1m)
rgb_1m = np.stack([gray_disp, gray_disp, gray_disp], axis=-1)

visualise_result(
    rgb_1m, pred_cart, names_cart, colors_cart,
    'Cartosat ResNet-18 — Panchromatic Test (1m)',
    save_path=os.path.join(OUT_DIR, 'test_1m_cartosat.png')
)

print('\nClass Area Statistics (res=1.0m):')
print_class_stats(pred_cart, names_cart, res_m=1.0)

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1m_cartosat.png

Class Area Statistics (res=1.0m):
Class                  Pixels Coverage    Area (m2) Area (km2)
-----------------------------------------------------------------
Low_Vegetation_Fields 19,445,052    98.1%   19,445,052    19.4451
Beaches_Sea_Shores       31,008     0.2%       31,008     0.0310
Roads_and_Pavement            0     0.0%            0     0.0000
Urban_Built_Up          346,560     1.7%      346,560     0.3466
Deep_Water_or_Shadows          0     0.0%            0     0.0000
Bare_Earth_Exposed_Soil          0     0.0%            0     0.0000


In [92]:
# Confidence metrics for Cartosat ResNet-18
print_confidence_metrics(pred_cart, conf_cart, names_cart, colors_cart)
plot_entropy_and_confidence(pred_cart, conf_cart, names_cart, colors_cart,
    'Cartosat ResNet-18 Confidence Analysis (1m PAN)',
    save_path=os.path.join(OUT_DIR, 'test_1m_cartosat_confidence.png'))
plot_class_distribution_pie(pred_cart, names_cart, colors_cart,
    'Cartosat ResNet-18 Class Distribution')

  CONFIDENCE METRICS (based on softmax probabilities)

Class                  Pixels  Mean Conf     Min     Max     Std    <50%    <70%
--------------------------------------------------------------------------------
Low_Vegetation_Fields 19,445,052      83.5%   35.5%   99.4%   12.7%    0.6%   35.9%
Beaches_Sea_Shores     31,008      61.6%   44.4%   80.1%   13.3%   40.6%   70.3%
Urban_Built_Up        346,560      64.7%   42.6%   89.0%   12.8%   13.3%   66.5%

Overall mean confidence: 83.1%
Pixels below 50% confidence: 0.9%
Pixels below 70% confidence: 36.5%
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1m_cartosat_confidence.png


In [93]:
# Export vectors
print('Generating vector shapefile...')
gdf_cart = raster_to_vectors(
    pred_cart, names_cart, colors_cart,
    geo_meta=geo_meta_1m
)
print(f'Total polygons: {len(gdf_cart)}')
print(gdf_cart.groupby('class_name').size().to_string())

shp_path = os.path.join(OUT_DIR, 'test_1m_vectors.zip')
shp_bytes = vectors_to_shapefile_zip(gdf_cart)
with open(shp_path, 'wb') as f:
    f.write(shp_bytes)
print(f'Saved: {shp_path} ({len(shp_bytes)/1024:.0f} KB)')

tiff_bytes = pred_map_to_geotiff_bytes(pred_cart, geo_meta=geo_meta_1m)
tiff_path = os.path.join(OUT_DIR, 'test_1m_prediction.tif')
with open(tiff_path, 'wb') as f:
    f.write(tiff_bytes)
print(f'Saved: {tiff_path}')

Generating vector shapefile...
Total polygons: 37
class_name
Beaches_Sea_Shores     4
Urban_Built_Up        33
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1m_vectors.zip (3 KB)
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1m_prediction.tif


---
## 4. Test: U-Net Vashi (Sentinel-2 style, on 2.5m panchromatic)

The Vashi U-Net expects 4-band input (B, G, R, NIR). Since `test_2_5m` is panchromatic (1 band),
the inference pipeline will replicate the band 4 times. The water heuristic will also activate
to correct dark-area misclassification.

In [94]:
print('Loading test_2_5m...')
with rasterio.open(os.path.join(TEST_DIR, 'test_2_5m')) as src:
    img_2_5m = src.read(1).astype(np.float32)
    geo_meta_2_5m = {
        'crs_wkt':   src.crs.to_wkt() if src.crs else None,
        'transform': tuple(src.transform),
        'width':     src.width,
        'height':    src.height,
    }

print(f'Shape: {img_2_5m.shape}  Range: {img_2_5m.min():.0f} - {img_2_5m.max():.0f}')
print(f'CRS: {src.crs}')

img_2_5m_hwc = img_2_5m[:, :, np.newaxis]

plt.figure(figsize=(10, 8))
plt.imshow(pct_stretch(img_2_5m), cmap='gray')
plt.title('Test Image — 2.5m Panchromatic', fontsize=13)
plt.axis('off')
plt.show()

Loading test_2_5m...
Shape: (1608, 1973)  Range: -32768 - 390
CRS: EPSG:32643


In [95]:
# Load model
print('Loading U-Net Vashi (Sentinel-2) model...')
model_vashi, ckpt_vashi = load_unet_vashi()
print(f'  Classes: {ckpt_vashi.get("num_classes")}  Input channels: {ckpt_vashi.get("in_channels")}')

# Run inference (1-band input -> pipeline replicates to 4 bands + applies water heuristic)
print('Running inference (band replication: 1 -> 4 channels)...')
result_vashi = run_unet(model_vashi, ckpt_vashi, img_2_5m_hwc.copy(), 'unet_vashi')

pred_vashi = result_vashi['pred_map']
conf_vashi = result_vashi['conf_map']
ds_step_v  = result_vashi.get('downsample_step', 1)
print(f'Prediction shape: {pred_vashi.shape}  (downsample_step={ds_step_v})')
print(f'Unique classes predicted: {np.unique(pred_vashi)}')
print(f'Mean confidence: {conf_vashi.mean()*100:.1f}%')

Loading U-Net Vashi (Sentinel-2) model...
  Classes: 7  Input channels: 4
Running inference (band replication: 1 -> 4 channels)...
Prediction shape: (1608, 1973)  (downsample_step=1)
Unique classes predicted: [0 1 2 3 4 5 6]
Mean confidence: 22.2%


In [96]:
# Visualise
names_vashi  = result_vashi['class_names']
colors_vashi = result_vashi['class_colors']

gray_disp_v = pct_stretch(img_2_5m)
rgb_2_5m = np.stack([gray_disp_v, gray_disp_v, gray_disp_v], axis=-1)

visualise_result(
    rgb_2_5m, pred_vashi, names_vashi, colors_vashi,
    'U-Net Vashi — Panchromatic Test (2.5m, band-replicated to 4ch)',
    save_path=os.path.join(OUT_DIR, 'test_2_5m_unet_vashi.png')
)

print('\nClass Area Statistics (res=2.5m):')
print_class_stats(pred_vashi, names_vashi, res_m=2.5)

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_2_5m_unet_vashi.png

Class Area Statistics (res=2.5m):
Class                  Pixels Coverage    Area (m2) Area (km2)
-----------------------------------------------------------------
Ignore                   95,469     3.0%      596,681     0.5967
Water                   366,058    11.5%    2,287,862     2.2879
Buildings               593,841    18.7%    3,711,506     3.7115
Roads                    54,322     1.7%      339,512     0.3395
Vegetation            1,999,596    63.0%   12,497,475    12.4975
Open Ground              14,912     0.5%       93,200     0.0932
Other Land               48,386     1.5%      302,412     0.3024


In [97]:
# Confidence metrics for U-Net Vashi
print_confidence_metrics(pred_vashi, conf_vashi, names_vashi, colors_vashi)
plot_entropy_and_confidence(pred_vashi, conf_vashi, names_vashi, colors_vashi,
    'U-Net Vashi Confidence Analysis (2.5m PAN)',
    save_path=os.path.join(OUT_DIR, 'test_2_5m_vashi_confidence.png'))
plot_class_distribution_pie(pred_vashi, names_vashi, colors_vashi,
    'U-Net Vashi Class Distribution')

  CONFIDENCE METRICS (based on softmax probabilities)

Class                  Pixels  Mean Conf     Min     Max     Std    <50%    <70%
--------------------------------------------------------------------------------
Ignore                 95,469      18.6%   14.3%   72.0%    4.3%   99.8%  100.0%
Water                 366,058      25.3%   14.6%   98.5%   10.0%   96.8%   98.6%
Buildings             593,841      23.2%   15.6%   99.8%    8.7%   97.9%   99.5%
Roads                  54,322      47.6%   16.1%   98.4%   19.3%   57.0%   83.4%
Vegetation           1,999,596      20.8%   15.6%   65.2%    3.4%  100.0%  100.0%
Open Ground            14,912      30.3%   16.0%   87.9%   11.3%   95.5%   97.5%
Other Land             48,386      22.5%   14.8%   78.2%    5.8%   99.2%  100.0%

Overall mean confidence: 22.2%
Pixels below 50% confidence: 98.4%
Pixels below 70% confidence: 99.4%
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_2_5m_vashi_confidence.png


In [98]:
# Export vectors
print('Generating vector shapefile...')
gdf_vashi = raster_to_vectors(
    pred_vashi, names_vashi, colors_vashi,
    geo_meta=geo_meta_2_5m, downsample_step=ds_step_v
)
print(f'Total polygons: {len(gdf_vashi)}')
print(gdf_vashi.groupby('class_name').size().to_string())

shp_path = os.path.join(OUT_DIR, 'test_2_5m_vectors.zip')
shp_bytes = vectors_to_shapefile_zip(gdf_vashi)
with open(shp_path, 'wb') as f:
    f.write(shp_bytes)
print(f'Saved: {shp_path} ({len(shp_bytes)/1024:.0f} KB)')

tiff_bytes = pred_map_to_geotiff_bytes(pred_vashi, geo_meta=geo_meta_2_5m, downsample_step=ds_step_v)
tiff_path = os.path.join(OUT_DIR, 'test_2_5m_prediction.tif')
with open(tiff_path, 'wb') as f:
    f.write(tiff_bytes)
print(f'Saved: {tiff_path}')

Generating vector shapefile...
Total polygons: 223758
class_name
Buildings      131831
Open Ground      3721
Other Land      14296
Roads            2654
Vegetation      46739
Water           24517
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_2_5m_vectors.zip (11198 KB)
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_2_5m_prediction.tif


---
## 5. Bonus: 4-Band Test (1.6m with all 4 channels → Vashi U-Net)

Stack all 4 bands of `test_1_6mc1-c4` and run the Vashi U-Net (which expects 4-band NIR input).
This tests whether the 4-band aerial data can be treated like Sentinel-2.

In [99]:
print('Loading all 4 bands of test_1_6m...')
bands_4ch = []
for band_name in ['test_1_6mc1', 'test_1_6mc2', 'test_1_6mc3', 'test_1_6mc4']:
    with rasterio.open(os.path.join(TEST_DIR, band_name)) as src:
        bands_4ch.append(src.read(1).astype(np.float32))

img_4ch = np.stack(bands_4ch, axis=-1)
print(f'4-band stack: {img_4ch.shape}')

# Run Vashi U-Net on 4-band input
print('Running Vashi U-Net on 4-band 1.6m data...')
result_4ch = run_unet(model_vashi, ckpt_vashi, img_4ch.copy(), 'unet_vashi')

pred_4ch = result_4ch['pred_map']
ds_step_4 = result_4ch.get('downsample_step', 1)
print(f'Prediction shape: {pred_4ch.shape}  (downsample_step={ds_step_4})')
print(f'Unique classes: {np.unique(pred_4ch)}')
print(f'Mean confidence: {result_4ch["conf_map"].mean()*100:.1f}%')

# Visualise
visualise_result(
    rgb_1_6m, pred_4ch, names_vashi, colors_vashi,
    'U-Net Vashi on 4-band Aerial (1.6m) — Cross-sensor Test',
    save_path=os.path.join(OUT_DIR, 'test_1_6m_4band_vashi.png')
)

print('\nClass Area Statistics (res=1.6m):')
print_class_stats(pred_4ch, names_vashi, res_m=1.6)

Loading all 4 bands of test_1_6m...
4-band stack: (1583, 2220, 4)
Running Vashi U-Net on 4-band 1.6m data...
Prediction shape: (792, 1110)  (downsample_step=2)
Unique classes: [0 1 2 3 4 5 6]
Mean confidence: 24.1%
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\test_1_6m_4band_vashi.png

Class Area Statistics (res=1.6m):
Class                  Pixels Coverage    Area (m2) Area (km2)
-----------------------------------------------------------------
Ignore                   49,744     5.7%      127,345     0.1273
Water                    68,762     7.8%      176,031     0.1760
Buildings               296,071    33.7%      757,942     0.7579
Roads                     8,999     1.0%       23,037     0.0230
Vegetation              430,121    48.9%    1,101,110     1.1011
Open Ground               2,371     0.3%        6,070     0.0061
Other Land               23,052     2.6%       59,013     0.0590


---
## 6. Accuracy Evaluation (KMeans Pseudo Ground Truth — Training-Matched)

Since we have no manual ground truth labels, we generate pseudo labels using KMeans clustering **with the same feature extraction pipelines used during model training**:

| Model | Training Clustering | Test Clustering (replicated) |
|-------|-------------------|------------------------------|
| **U-Net GBNM** | KMeans on [B1, B2, B3, Brightness, ExG, NGRDI] + StandardScaler | Same spectral indices |
| **Cartosat ResNet-18** | KMeans on ResNet18 features + PCA(50) | Multi-scale texture features (mean/std at 3/7/15px + gradient) |
| **U-Net Vashi** | Manual ground truth (supervised) | Texture-based clustering |

We over-cluster with **20 clusters** then use **majority-vote mapping** (many-to-one) to map clusters to predicted classes. This captures spectral sub-classes properly.

In [100]:
# Force-reload labeling_utils
import importlib
import labeling_utils as _lu
importlib.reload(_lu)
from labeling_utils import (generate_kmeans_gt, generate_spectral_kmeans_gt,
    generate_texture_kmeans_gt, smooth_label_map,
    majority_vote_match, compute_full_metrics, remap_labels,
    print_accuracy_report, plot_confusion_matrix, plot_gt_vs_pred,
    plot_model_comparison, print_comparison_table, downsample_image)

N_OVER = 50
SMOOTH_KERNEL = 15

# === Super-class merge maps (applied AFTER GT generation) ===
# Super-classes: Water(0), Vegetation(1), Built-up(2)

# GBNM: 0=Background, 1=Water, 2=Vegetation, 3=Roads, 4=Building, 5=OpenGround
GBNM_MERGE = {0: 2, 1: 0, 2: 1, 3: 2, 4: 2, 5: 2}
# Cartosat: 0=LowVeg, 1=Beaches, 2=Roads, 3=Urban, 4=DeepWater, 5=BareEarth
CART_MERGE = {0: 1, 1: 0, 2: 2, 3: 2, 4: 0, 5: 2}
# Vashi: 0=Ignore, 1=Water, 2=Buildings, 3=Roads, 4=Vegetation, 5=OpenGround, 6=OtherLand
VASHI_MERGE = {0: 2, 1: 0, 2: 2, 3: 2, 4: 1, 5: 2, 6: 2}

SUPER_NAMES = ['Water', 'Vegetation', 'Built-up']
SUPER_COLORS = ['#0077be', '#228b22', '#d2691e']

print('Generating KMeans pseudo ground truth labels...')
print(f'  Clusters: {N_OVER}, Smoothing: {SMOOTH_KERNEL}x{SMOOTH_KERNEL}')
print(f'  Step 1: GT with fine-grained classes via majority vote')
print(f'  Step 2: Remap both pred & GT to 3 super-classes\n')

import gc

# 1) GBNM — generate GT with original 6 classes, then remap
print('--- 1.6m RGB: Spectral-index clustering (GBNM) ---')
cluster_1_6m, _ = generate_spectral_kmeans_gt(img_1_6m, n_clusters=N_OVER, nodata_val=-32768)
cluster_1_6m_ds = downsample_image(cluster_1_6m, pred_gbnm.shape)
gt_1_6m_raw, mapping_gbnm = majority_vote_match(pred_gbnm, cluster_1_6m_ds, n_classes=6)
gt_1_6m_raw = smooth_label_map(gt_1_6m_raw, kernel_size=SMOOTH_KERNEL)
# Show fine-grained distribution
dist_fine = {names_gbnm[c]: sum(1 for v in mapping_gbnm.values() if v==c) for c in sorted(set(mapping_gbnm.values()))}
print(f'  Fine-grained: {dist_fine}')
# Remap to super-classes
gt_1_6m_gbnm = remap_labels(gt_1_6m_raw, GBNM_MERGE)
pred_gbnm_merged = remap_labels(pred_gbnm, GBNM_MERGE)
for sc in range(3):
    gt_pct = 100*(gt_1_6m_gbnm==sc).sum()/max((gt_1_6m_gbnm!=255).sum(),1)
    pr_pct = 100*(pred_gbnm_merged==sc).sum()/pred_gbnm_merged.size
    print(f'  {SUPER_NAMES[sc]:12s}: GT={gt_pct:.1f}%, Pred={pr_pct:.1f}%')
del cluster_1_6m, cluster_1_6m_ds, gt_1_6m_raw; gc.collect()

# 2) Cartosat
print('\n--- 1m PAN: Texture clustering (Cartosat) ---')
cluster_1m, _ = generate_texture_kmeans_gt(img_1m, n_clusters=N_OVER, nodata_val=-32768)
cluster_1m_ds = downsample_image(cluster_1m, pred_cart.shape)
gt_1m_raw, mapping_cart = majority_vote_match(pred_cart, cluster_1m_ds, n_classes=6)
gt_1m_raw = smooth_label_map(gt_1m_raw, kernel_size=SMOOTH_KERNEL)
dist_fine = {names_cart[c]: sum(1 for v in mapping_cart.values() if v==c) for c in sorted(set(mapping_cart.values()))}
print(f'  Fine-grained: {dist_fine}')
gt_1m_cart = remap_labels(gt_1m_raw, CART_MERGE)
pred_cart_merged = remap_labels(pred_cart, CART_MERGE)
for sc in range(3):
    gt_pct = 100*(gt_1m_cart==sc).sum()/max((gt_1m_cart!=255).sum(),1)
    pr_pct = 100*(pred_cart_merged==sc).sum()/pred_cart_merged.size
    print(f'  {SUPER_NAMES[sc]:12s}: GT={gt_pct:.1f}%, Pred={pr_pct:.1f}%')
del cluster_1m, cluster_1m_ds, gt_1m_raw; gc.collect()

# 3) Vashi
print('\n--- 2.5m PAN: Texture clustering (Vashi) ---')
cluster_2_5m, _ = generate_texture_kmeans_gt(img_2_5m, n_clusters=N_OVER, nodata_val=-32768)
cluster_2_5m_ds = downsample_image(cluster_2_5m, pred_vashi.shape)
gt_2_5m_raw, mapping_vashi = majority_vote_match(pred_vashi, cluster_2_5m_ds, n_classes=7)
gt_2_5m_raw = smooth_label_map(gt_2_5m_raw, kernel_size=SMOOTH_KERNEL)
dist_fine = {names_vashi[c]: sum(1 for v in mapping_vashi.values() if v==c) for c in sorted(set(mapping_vashi.values()))}
print(f'  Fine-grained: {dist_fine}')
gt_2_5m_vashi = remap_labels(gt_2_5m_raw, VASHI_MERGE)
pred_vashi_merged = remap_labels(pred_vashi, VASHI_MERGE)
for sc in range(3):
    gt_pct = 100*(gt_2_5m_vashi==sc).sum()/max((gt_2_5m_vashi!=255).sum(),1)
    pr_pct = 100*(pred_vashi_merged==sc).sum()/pred_vashi_merged.size
    print(f'  {SUPER_NAMES[sc]:12s}: GT={gt_pct:.1f}%, Pred={pr_pct:.1f}%')
del cluster_2_5m, cluster_2_5m_ds, gt_2_5m_raw; gc.collect()

print('\nDone.')

Generating KMeans pseudo ground truth labels...
  Clusters: 50, Smoothing: 15x15
  Step 1: GT with fine-grained classes via majority vote
  Step 2: Remap both pred & GT to 3 super-classes

--- 1.6m RGB: Spectral-index clustering (GBNM) ---
  Fine-grained: {'Vegetation': 50}
  Water       : GT=0.0%, Pred=16.4%
  Vegetation  : GT=100.0%, Pred=67.4%
  Built-up    : GT=0.0%, Pred=16.2%

--- 1m PAN: Texture clustering (Cartosat) ---
  Fine-grained: {'Low_Vegetation_Fields': 50}
  Water       : GT=0.0%, Pred=0.2%
  Vegetation  : GT=100.0%, Pred=98.1%
  Built-up    : GT=0.0%, Pred=1.7%

--- 2.5m PAN: Texture clustering (Vashi) ---
  Fine-grained: {'Buildings': 4, 'Vegetation': 46}
  Water       : GT=0.0%, Pred=11.5%
  Vegetation  : GT=99.7%, Pred=63.0%
  Built-up    : GT=0.3%, Pred=25.4%

Done.


### 6a. U-Net GBNM Accuracy (1.6m RGB)

In [101]:
# GBNM evaluation with 3 super-classes
metrics_gbnm = compute_full_metrics(pred_gbnm_merged, gt_1_6m_gbnm, GBNM_SUPER_NAMES)
print_accuracy_report(metrics_gbnm, 'U-Net GBNM (1.6m Aerial RGB) — 3-class evaluation')
plot_gt_vs_pred(gt_1_6m_gbnm, pred_gbnm_merged, GBNM_SUPER_NAMES, GBNM_SUPER_COLORS,
    'U-Net GBNM: KMeans GT vs Prediction (1.6m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_gbnm_gt_vs_pred.png'))
plot_confusion_matrix(metrics_gbnm, GBNM_SUPER_NAMES,
    'U-Net GBNM Confusion Matrix (1.6m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_gbnm_confusion.png'))

  U-Net GBNM (1.6m Aerial RGB) — 3-class evaluation

  Overall Accuracy (OA):  69.65%
  Mean IoU (mIoU):        23.22%
  Freq-Weighted IoU:      69.65%
  Cohen Kappa:            0.0000

  Class                 Precision     Recall   F1-Score        IoU    Support
  ------------------------------------------------------------------------
  Water                      0.0%       0.0%       0.0%       0.0%          0
  Vegetation               100.0%      69.7%      82.1%      69.7%     842872
  Built-up                   0.0%       0.0%       0.0%       0.0%          0

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_gbnm_gt_vs_pred.png
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_gbnm_confusion.png


### 6b. Cartosat ResNet-18 Accuracy (1m PAN)

In [102]:
# Cartosat evaluation with 3 super-classes
metrics_cart = compute_full_metrics(pred_cart_merged, gt_1m_cart, CART_SUPER_NAMES)
print_accuracy_report(metrics_cart, 'Cartosat ResNet-18 (1m Panchromatic) — 3-class evaluation')
plot_gt_vs_pred(gt_1m_cart, pred_cart_merged, CART_SUPER_NAMES, CART_SUPER_COLORS,
    'Cartosat ResNet-18: KMeans GT vs Prediction (1m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_cartosat_gt_vs_pred.png'))
plot_confusion_matrix(metrics_cart, CART_SUPER_NAMES,
    'Cartosat ResNet-18 Confusion Matrix (1m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_cartosat_confusion.png'))

  Cartosat ResNet-18 (1m Panchromatic) — 3-class evaluation

  Overall Accuracy (OA):  98.94%
  Mean IoU (mIoU):        32.98%
  Freq-Weighted IoU:      98.94%
  Cohen Kappa:            0.0000

  Class                 Precision     Recall   F1-Score        IoU    Support
  ------------------------------------------------------------------------
  Water                      0.0%       0.0%       0.0%       0.0%          0
  Vegetation               100.0%      98.9%      99.5%      98.9%   12495577
  Built-up                   0.0%       0.0%       0.0%       0.0%          0

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_cartosat_gt_vs_pred.png
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_cartosat_confusion.png


### 6c. U-Net Vashi Accuracy (2.5m PAN)

In [103]:
# Vashi evaluation with 3 super-classes
metrics_vashi = compute_full_metrics(pred_vashi_merged, gt_2_5m_vashi, VASHI_SUPER_NAMES)
print_accuracy_report(metrics_vashi, 'U-Net Vashi (2.5m Panchromatic) — 3-class evaluation')
plot_gt_vs_pred(gt_2_5m_vashi, pred_vashi_merged, VASHI_SUPER_NAMES, VASHI_SUPER_COLORS,
    'U-Net Vashi: KMeans GT vs Prediction (2.5m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_vashi_gt_vs_pred.png'))
plot_confusion_matrix(metrics_vashi, VASHI_SUPER_NAMES,
    'U-Net Vashi Confusion Matrix (2.5m)',
    save_path=os.path.join(OUT_DIR, 'accuracy_vashi_confusion.png'))

  U-Net Vashi (2.5m Panchromatic) — 3-class evaluation

  Overall Accuracy (OA):  75.70%
  Mean IoU (mIoU):        25.49%
  Freq-Weighted IoU:      75.40%
  Cohen Kappa:            0.0097

  Class                 Precision     Recall   F1-Score        IoU    Support
  ------------------------------------------------------------------------
  Water                      0.0%       0.0%       0.0%       0.0%          0
  Vegetation                99.8%      75.8%      86.1%      75.7%    1992508
  Built-up                   0.8%      58.3%       1.6%       0.8%       6787

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_vashi_gt_vs_pred.png
Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_vashi_confusion.png


### 6d. Model Comparison — Which performs best?

In [104]:
all_metrics = [
    {'name': 'U-Net GBNM (1.6m RGB)', 'metrics': metrics_gbnm},
    {'name': 'Cartosat ResNet-18 (1m PAN)', 'metrics': metrics_cart},
    {'name': 'U-Net Vashi (2.5m PAN)', 'metrics': metrics_vashi},
]

print_comparison_table(all_metrics)
plot_model_comparison(all_metrics,
    save_path=os.path.join(OUT_DIR, 'accuracy_model_comparison.png'))

  MODEL COMPARISON SUMMARY

  Model                                OA (%)     mIoU (%)    FWIoU (%)        Kappa  Best Class IoU
  ----------------------------------------------------------------------------------------------------
  U-Net GBNM (1.6m RGB)                69.65%       23.22%       69.65%       0.0000 Vegetation (69.7%)
  Cartosat ResNet-18 (1m PAN)          98.94%       32.98%       98.94%       0.0000 Vegetation (98.9%)
  U-Net Vashi (2.5m PAN)               75.70%       25.49%       75.40%       0.0097 Vegetation (75.7%)

  >>> Best performing model: Cartosat ResNet-18 (1m PAN) (OA = 98.94%)

Saved: c:\Users\PRABHAKAR\Documents\Feature-Ext(Prabhakar)\Test_Model\results\accuracy_model_comparison.png


---
## 7. Summary — All Test Outputs

In [105]:
print('=' * 70)
print('  TEST COMPLETE — ALL OUTPUT FILES')
print('=' * 70)

for f in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, f)
    size = os.path.getsize(fpath)
    if size > 1e6:
        size_str = f'{size/1e6:.1f} MB'
    else:
        size_str = f'{size/1024:.0f} KB'
    print(f'  {f:45s}  {size_str:>10s}')

print(f'\nAll outputs in: {OUT_DIR}')
print('\nShapefiles can be opened in QGIS / ArcGIS for spatial analysis.')
print('GeoTIFFs can be overlaid on basemaps with correct georeferencing.')

  TEST COMPLETE — ALL OUTPUT FILES
  accuracy_cartosat_confusion.png                     78 KB
  accuracy_cartosat_gt_vs_pred.png                   110 KB
  accuracy_gbnm_confusion.png                         78 KB
  accuracy_gbnm_gt_vs_pred.png                       1.5 MB
  accuracy_model_comparison.png                       60 KB
  accuracy_vashi_confusion.png                        80 KB
  accuracy_vashi_gt_vs_pred.png                      1.9 MB
  test_1_6m_4band_vashi.png                          1.9 MB
  test_1_6m_gbnm_confidence.png                      1.3 MB
  test_1_6m_prediction.tif                           124 KB
  test_1_6m_unet_gbnm.png                            2.2 MB
  test_1_6m_vectors.zip                              2.6 MB
  test_1m_cartosat.png                               121 KB
  test_1m_cartosat_confidence.png                    111 KB
  test_1m_prediction.tif                             523 KB
  test_1m_vectors.zip                                  3 KB
  tes